To make llms run efficiently, you need to understand attention. Not just the architecture, but the memory and FLOPs needed for attention. In other words, you have to be a FLOPhead.

In this post, we'll cover the differences between multi-headed attention (MHA), group query attention (GQA), and multi-headed latent attention (MLA) from a MLsys perspective. We are going to count FLOPs and FLOPS (there is a difference), compute arithmetic intensity, and find out why different versions of attention are more efficient than others. This post will use tinygrad, a new competitor to pytorch, that makes it infinitely easier to peer underneath the hood and access the memory and FLOPs of your model.

Let's dive in.

# Intro

FLOPs or **FL**oating **P**oint **O**perations represent the number of floating point arithmetic operations (such as scalar additions, subtractions, multiplications, divisions) performed by a neural network. FLOPs is metric that quantifies how much computation is performed. An A100 GPU can perform FLOPs per second, this is known as FLOPS. FLOPs with the lowercase s is the plural of FLOP and is a measure of *compute*; FLOPS with the capital S means FLOPs per second and is a measure of *compute speed*. By knowing the FLOPs of different parts of your model, you can identify computationally expensive operations and optimize them.

Multiplying a vector of length $n$ by $7$ requires $n$ FLOPs because we are performing $n$ element-wise multiplications.

Computing the dot product between two vectors of length $n$ takes $2n-1 \approx 2n$ FLOPs: $n$ element-wise multiplications and $n-1$ element-wise additions.

Multiplying a $(m, n)$ matrix by a $(n, p)$ matrix requires $mp (2n-1) \approx 2mpn$ FLOPs: for each of the $mp$ entries of the resultant matrix we compute a dot product at the cost of $2n-1$ ops. (Yes, matrix multiplication can just be thought of as a bunch of dot products.)

We can generalize this to tensors. Multiplying a $(a, m, n)$ tensor (yes, this has three dimensions) by a $(n, p)$ tensor requires $a mp (2n-1) \approx 2ampn$ FLOPs: imagine we are performing matrix multiplication between a $(m, n)$ matrix by a $(n, p)$ matrix $a$ different times, each time costing $mp (2n-1)$ FLOPs.

To compute FLOPs, we just need to count. It is pretty simple.

Yet it is pretty much impossible to accurately count FLOPs in pytorch. This remarkably simple feature simply does not exist in pytorch even though it has been [requested](https://github.com/pytorch/pytorch/issues/5013) since 2025.

![](pytorch_flop_counter_request.png)

The pytorch team [implemented](https://github.com/pytorch/pytorch/blob/660e369a68dd8be60ce4eb67c25191ea66efc303/torch/utils/flop_counter.py#L617) a poorly documented FLOP counter, `FlopCounterMode`. But this feature is totally useless. `FlopCounterMode` ignores element-wise ops like `sin`, `add`, `mul`, cannot handle torch.compile, breaks on custom kernels, and incurs such a significant overhead that renders it utterly unusable in many cases. See more complaints in [this](https://github.com/pytorch/pytorch/issues/123800#issue-2236772575) github discussion. One of the pytorch GOATs, [Horace He](https://horace.io/), [implemented](https://dev-discuss.pytorch.org/t/the-ideal-pytorch-flop-counter-with-torch-dispatch/505) his own hackable flops counter, but it also has significant overhead of 30-40us per operator. The pytorch profiler doesn't track FLOPs out of the box either. Simply put, pytorch is [collapsing](https://neel04.github.io/my-website/blog/pytorch_rant/) under it's own weight and cannot support a simple FLOP counter that is 100% accurate. If anyone knows how to *exactly* measure FLOPs in pytorch, please tell me! I'd love to learn how.

There are [numerous](https://github.com/Lyken17/pytorch-OpCounter) [libraries](https://github.com/sovrasov/flops-counter.pytorch) [dedicated](https://github.com/facebookresearch/fvcore/blob/main/docs/flop_count.md) [to](https://github.com/TylerYep/torchinfo) counting FLOPs in pytorch, many of which have thousands of github stars. Entire libraries just for counting! And they still are not always 100% correct.

There needs to be a simpler way.

Enter [tinygrad](https://github.com/tinygrad/tinygrad).

tinygrad is one of the most exciting open source AI projects. It is built as a competitor to pytorch that is infinitely simpler. The hope is that the simplicity results in faster operations and cheaper, more efficient models. It also uses some really cool compiler ideas.

In tinygrad, you use the `GlobalCounters` to easily inspect the FLOPs

In [1]:
from tinygrad import Tensor, nn
from tinygrad.helpers import GlobalCounters

Adding two vectors of length 3 should take 3 FLOPs.

In [2]:
# record flops seen so far
FLOPs0 = GlobalCounters.global_ops

# add two tensors
t = Tensor([1, 2, 3]) + Tensor([1, 2, 3])

# record the vector addition flops
assert GlobalCounters.global_ops - FLOPs0 == 0

Wait, the assert tells us it takes *zero FLOPs*, not 3 FLOPs, to add these two vectors.

This is because tinygrad in lazy. In tinygrad, no computation is actually performed until we call `Tensor.realize()`. This is why we 0 FLOPs are performed. Now let's realize the computation.

In [3]:
# actually perform the computation
t.realize()
# record the new number of flops
assert GlobalCounters.global_ops - FLOPs0 == 3

After calling `.realize()`, we get the 3 FLOPs, we expected! 

Let's try a mat-mul with a batch dimension.

In [ ]:
# initialize two matrices
t1 = Tensor.arange(12).reshape(3, 2, 2).realize() # (B,T,D)
t2 = Tensor.arange(12).reshape(2, 6).realize() # (D,D2)

# record flops seen so far
FLOPs0 = GlobalCounters.global_ops

# multiply two matrices
# actually perform the computation with .realize()
t3 = t1.matmul(t2).realize()

# record the mat mul flops
assert GlobalCounters.global_ops - FLOPs0 == 108

This matrix multiplication takes 108 FLOPs. Why?

Multiplying a `(3, 2, 2)` tensor by a `(2, 6)` matrix takes $amp(2n-1) = 3*2*6*(2*2 - 1) = 108$ FLOPs. This uses the exact equation we used before for tensor multiplication.

Lastly, let's count the FLOPs of a small model with a linear layer and a convolution:

In [5]:
# define the model
class Model:
    def __init__(self): self.l1, self.conv = nn.Linear(3, 3, bias=False), nn.Conv1d(3, 1, 1, bias=False)
    def __call__(self, x): return self.conv(self.l1(x))

# realize the model parameters
model = Model()
Tensor.realize(*nn.state.get_parameters(model))

# create input and realize it
x = Tensor.arange(4*3*3).reshape(4, 3, 3).realize()

# record flops seen so far
FLOPs0 = GlobalCounters.global_ops

# forward pass of the model
# actually perform the computation with .realize()
out = model(x).realize()

# record the forward pass flops
assert GlobalCounters.global_ops - FLOPs0 == 240

Our model takes 240 FLOPs.

The linear layer performs a matrix multiply between the input `x.shape = (4,3,3)` and `self.l1.weight.shape = (3,3)` which takes $a \cdot m \cdot p \cdot (2n-1) = 4 \cdot 3 \cdot 3 \cdot (2 \cdot 3 - 1) = 250$ FLOPs. 

The convolution layer performs a 1D convolution with `self.conv.weight.shape = (1,3,1)` (1 output channel, 3 input channels, kernel size 1). For each output element, we compute a dot product across 3 input channels and 1 kernel position, requiring $3 \cdot 1 \cdot 2 - 1 = 5$ FLOPs per output. With an output shape of `(4,1,3)` (4 batches, 1 output channel, 3 positions), we have $4 \cdot 1 \cdot 3 = 12$ output elements. This gives us $12 \cdot 5 = 60$ FLOPs for the convolution.

In total, the forward pass requires $250 + 60 = 240$ FLOPs.

We can use this to build a simple helper that prints out the FLOPs and other helpful statistics.

In [2]:
import contextlib, math, time
from tinygrad import Tensor, nn, dtypes
from tinygrad.helpers import GlobalCounters, colored

In [7]:
def time_to_str(t:float, w=18) -> str: return colored(next((f"{t * d:{w-len(pr)-1}.2f} {pr}" for d,pr in [(1, "s "),(1e3, "ms")] if t > 10/d), f"{t * 1e6:{w-4}.2f} us"), 'yellow')
def op_to_str(op:int, w=18) -> str: return colored(next((f"{op / d:{w-len(pr)-1}.2f} {pr}" for d,pr in [(1e12, "TOPs"), (1e9, "GOPs")] if op >= d), f"{op:{w-5}} OPs"), 'yellow')
def mem_to_str(mem:int, w=18) -> str: return colored(next((f"{mem / d:{w-len(pr)-1}.2f} {pr}" for d,pr in [(1e9, "GB"), (1e6, "MB")] if mem >= d), f"{mem:{w-7}} bytes"), 'yellow')
def op_tm_to_str(x:float, w=25) -> str: return colored(next((f"{x / d:{w-len(pr)-1}.2f} {pr}" for d,pr in [(1e12, "TOPs/sec"), (1e9, "GOPs/sec"), (1e6, "MOPs/sec")] if x >= d), f"{x:{w-9}.2f} OPs/sec"), 'blue')
def mem_tm_to_str(x:float, w=25) -> str: return colored(next((f"{x / d:{w-len(pr)-1}.2f} {pr}" for d,pr in [(1e9, "GB/sec"), (1e6, "MB/sec")] if x >= d), f"{x:{w-11}.2f} bytes/sec"), 'blue')
def op_mem_to_str(x:float, w=25) -> str: return colored(next((f"{x / d:{w-len(pr)-1}.2f} {pr}" for d,pr in [(1e9, "GOPs/GB"), (1e6, "MOPs/MB")] if x >= d), f"{x:{w-10}.2f} OPs/byte"), 'blue')

In [15]:
class Stats(contextlib.ContextDecorator):
  def __init__(self, prefix="", enabled=True, header=False): self.prefix, self.enabled, self.header = prefix, enabled, header
  def __enter__(self): self.tm, self.op, self.mem = time.perf_counter_ns(), GlobalCounters.global_ops, GlobalCounters.mem_used # GlobalCounters.global_mem
  def __exit__(self, *exc):
    if not self.enabled: return
    if self.header: print(f"{'':40}{'Time':<25}  {'Compute (FLOPs)':<25}  {'Memory':<25}  {'Compute Bandwidth (FLOPS)':<25}  {'Memory Bandwidth':<25}  {'Arithmetic Intensity':<25}")
    tm, op, mem = (time.perf_counter_ns()-self.tm)*1e-9, GlobalCounters.global_ops-self.op, GlobalCounters.mem_used-self.mem
    op_tm, mem_tm, op_mem = op/(tm or 1e-20), mem/(tm or 1e-20), op/(mem or 1e-20)
    print(f"{self.prefix:<40}{time_to_str(tm)}, {op_to_str(op)}, {mem_to_str(mem)}, {op_tm_to_str(op_tm)}, {mem_tm_to_str(mem_tm)}, {op_mem_to_str(op_mem)}")

Let's use the helper on our model from before:

In [16]:
with Stats("Model Parameters:", header=True):
    model = Model()
    Tensor.realize(*nn.state.get_parameters(model))

with Stats("Model Activation:"):
    x = Tensor.arange(4*3*3).reshape(4, 3, 3).realize()

with Stats("Model Forward Pass:"):
    out = model(x).realize()

                                        Time                       Compute (FLOPs)            Memory                     Compute Bandwidth (FLOPS)  Memory Bandwidth           Arithmetic Intensity     
Model Parameters:                                 30.38 ms,          2759 OPs,           0 bytes,         90829.03 OPs/sec,           0.00 bytes/sec, 275900000000000.00 GOPs/GB
Model Activation:                              1260.79 us,           432 OPs,           0 bytes,        342642.04 OPs/sec,           0.00 bytes/sec, 43200000000000.00 GOPs/GB
Model Forward Pass:                            1567.17 us,           240 OPs,           0 bytes,        153142.58 OPs/sec,           0.00 bytes/sec, 24000000000000.00 GOPs/GB


We get the stats from
* Loading the model parameters. 
* 
We can very easily see the FLOPs needed to initialize the model

In [10]:
class MHA2:
    def __init__(self, dim:int n_heads:int):
        self.dim = dim
        self.n_heads = n_heads
        self.head_dim = dim // n_heads

        self.attn_q = nn.linear(dim, dim) # (D, D)
        self.attn_k = nn.linear(dim, dim) # (D, D)
        self.attn_v = nn.linear(dim, dim) # (D, D)
        self.attn_o = nn.linear(dim, dim) # (D, D)

    def __call__(self, x, causal=True):
        # projections
        q, k, v = self.attn_q(x), self.attn_k(x), self.attn_v(x) # (B,T,D) -> (B,T,D)

        # reshape
        B, T, _ = x.shape
        q = q.reshape(B, T, self.n_heads, self.head_dim).transpose(1, 2) # (B,T,D) -> (B,H,T,D_h)
        k = k.reshape(B, T, self.n_heads, self.head_dim).transpose(1, 2) # (B,T,D) -> (B,H,T,D_h)
        v = v.reshape(B, T, self.n_heads, self.head_dim).transpose(1, 2) # (B,T,D) -> (B,H,T,D_h)

        # compute attention
        qk = q @ k.transpose(-1, -2) / math.sqrt(q.shape[-1]) # (B,H,T,D_h) (B,H,D_h,T) -> (B,H,T,T)
        if causal:
            mask = qk.full_like(float("-inf")).triu(0)
            qk += mask
        smax = qk.softmax(-1) # (B,H,T,T) -> (B,H,T,T)
        attn = smax @ v # (B,H,T,T) (B,H,T,D_h) -> (B,H,T,D_h)
        attn = attn.transpose(1, 2).reshape(B,T,-1) # (B,H,T,D_h) -> (B,T,D)
        out = self.attn_o(attn) # (B,T,D) -> (B,T,D)
        return out




SyntaxError: invalid syntax. Perhaps you forgot a comma? (1730270090.py, line 2)

In [ ]:
class MHA3:
    def __init__(self, dim:int,  n_heads:int):
        self.dim = dim
        self.n_heads = n_heads
        self.head_dim = dim // n_heads

        self.attn_q = nn.Linear(dim, dim)
        self.attn_k = nn.Linear(dim, dim)
        self.attn_v = nn.Linear(dim, dim)
        self.attn_o = nn.Linear(dim, dim)

    def rope(self, x): return x

    def __call__(self, x:Tensor, causal=True):
        # projections
        q = self.attn_q(x) # (B,T,D) -> (B,T,D)
        k = self.attn_k(x) # (B,T,D) -> (B,T,D)
        v = self.attn_v(x) # (B,T,D) -> (B,T,D)
        print(f'{q.numpy()=}\n{k.numpy()=}\n{v.numpy()=}')

        # reshape
        B, T, _ = x.shape
        q = q.reshape(B, T, self.n_heads, self.head_dim).transpose(1, 2) # (B,H,T,Dh)
        k = k.reshape(B, T, self.n_heads, self.head_dim).transpose(1, 2) # (B,H,T,Dh)
        v = v.reshape(B, T, self.n_heads, self.head_dim).transpose(1, 2) # (B,H,T,Dh)
        print(f'{q.numpy()=}\n{k.numpy()=}\n{v.numpy()=}')

        # positional embedding
        q, k = self.rope(q), self.rope(k)

        # compute attention
        qk = q @ k.transpose(-1, -2) / math.sqrt(q.shape[-1]) # (B,H,T,Dh) (B,H,Dh,T) -> (B,H,T,T)
        print(f'{qk.numpy()=}')
        if causal:
            mask = Tensor.full_like(qk, float('-inf')).triu()
            qk = qk + mask
            print(f'{qk.numpy()=}')
        smax = qk.softmax(-1) # (B,H,T,T) -> (B,H,T,T)
        attn = smax @ v # (B,H,T,T) (B,H,T,Dh) -> (B,H,T,Dh)
        attn = attn.transpose(1, 2).reshape(B,T,-1) # (B,T,D)
        out = self.attn_o(attn) # (B,T,D) -> (B,T,D)
        return out


In [17]:
B, T, D, H = 2, 3, 2, 1
x = Tensor.arange(B*T*D).reshape(B,T,D).realize()
model = MHA3(D, H)
out = model(x).realize()
out.numpy()

q.numpy()=array([[[ 0.81988716, -0.3453666 ],
        [ 1.1380605 , -1.150455  ],
        [ 1.4562337 , -1.9555434 ]],

       [[ 1.7744071 , -2.7606318 ],
        [ 2.0925803 , -3.5657206 ],
        [ 2.4107535 , -4.370809  ]]], dtype=float32)
k.numpy()=array([[[-0.48522604,  1.1282945 ],
        [-1.5452166 ,  2.7609076 ],
        [-2.6052072 ,  4.393521  ]],

       [[-3.6651976 ,  6.0261335 ],
        [-4.7251883 ,  7.658747  ],
        [-5.7851787 ,  9.29136   ]]], dtype=float32)
v.numpy()=array([[[ 0.7731507 ,  0.6377916 ],
        [ 2.1844878 ,  0.22804292],
        [ 3.5958247 , -0.18170573]],

       [[ 5.007162  , -0.5914544 ],
        [ 6.418499  , -1.0012031 ],
        [ 7.829836  , -1.4109516 ]]], dtype=float32)
q.numpy()=array([[[[ 0.81988716, -0.3453666 ],
         [ 1.1380605 , -1.150455  ],
         [ 1.4562337 , -1.9555434 ]]],


       [[[ 1.7744071 , -2.7606318 ],
         [ 2.0925803 , -3.5657206 ],
         [ 2.4107535 , -4.370809  ]]]], dtype=float32)
k.numpy()=a

array([[[        nan,         nan],
        [ 1.0641435 ,  0.12562852],
        [ 1.0629239 ,  0.09095473]],

       [[        nan,         nan],
        [ 0.9563073 , -2.9404013 ],
        [ 0.9562694 , -2.9414802 ]]], dtype=float32)

# Multi-Headed Attention (MHA)

In [ ]:
def apply_rope(x:Tensor, start_pos:int, base:float = 10000.0) -> Tensor:
  B, H, T, Hd = x.shape
  assert (Hd & 1) == 0, "RoPE requires an even head dimension"
  half = Hd // 2
  angles = (Tensor.arange(T, dtype="float32") + start_pos)[:, None] * (base ** (-(Tensor.arange(half, dtype="float32") / half)))[None, :]
  cos, sin = angles.cos().reshape(1, 1, T, half).cast(x.dtype), angles.sin().reshape(1, 1, T, half).cast(x.dtype)
  x_pairs = x.reshape(B, H, T, half, 2)
  return Tensor.stack(x_pairs[..., 0] * cos - x_pairs[..., 1] * sin,
                      x_pairs[..., 0] * sin + x_pairs[..., 1] * cos, dim=-1).reshape(B, H, T, Hd)

In [ ]:
# multi-headed attention
# https://arxiv.org/abs/1706.03762
class MHA:
  def __init__(self, dim:int, num_heads:int, max_context:int=0):
    self.num_heads = num_heads # H
    self.head_dim = dim // num_heads # Dh
    self.max_context = max_context # T

    self.attn_q = nn.Linear(dim, dim, bias=False) # (D,D)
    self.attn_k = nn.Linear(dim, dim, bias=False) # (D,D)
    self.attn_v = nn.Linear(dim, dim, bias=False) # (D,D)
    self.attn_o = nn.Linear(dim, dim, bias=False) # (D,D)

  def __call__(self, x:Tensor, start_pos:int=0, is_causal=False, kv_cache=False, rope=False) -> Tensor:
    # projections
    q, k, v = self.attn_q(x), self.attn_k(x), self.attn_v(x) # (B,T,D) -> (B,T,D)

    # reshape
    B, T, _ = x.shape
    q = q.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)  # (B,T,D) -> (B,H,T,Dh)
    k = k.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)  # (B,T,D) -> (B,H,T,Dh)
    v = v.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)  # (B,T,D) -> (B,H,T,Dh)

    # positional embeddings
    if rope: q, k = apply_rope(q, start_pos), apply_rope(k, start_pos) # (B,H,T,Dh) -> (B,H,T,Dh)

    # kv cache
    if kv_cache and T > 1:
      if not hasattr(self, "kv_cache"):
        self.kv_cache = Tensor.zeros(2, self.max_context, B, self.num_heads, self.head_dim, dtype=k.dtype, device=k.device).contiguous().realize()
      self.kv_cache[:, start_pos:start_pos+T, :, :, :].assign(Tensor.stack(k, v))
      k = self.kv_cache[0, 0:start_pos+T, :, :, :]
      v = self.kv_cache[1, 0:start_pos+T, :, :, :]

    # compute attention
    qk = (q.matmul(k.transpose(-1, -2)) / math.sqrt(q.shape[-1])) # (B,H,T,Dh) (B,H,Dh,T) -> (B,H,T,T)
    if is_causal and T > 1:
      mask = Tensor.full((1, 1, T, start_pos+T), float("-inf"), dtype=x.dtype, device=x.device).triu(start_pos+1) # (B,H,T,T)
      qk = (qk + mask) # (B,H,T,T)
    s = qk.softmax(-1) # (B,H,T,T) -> (B,H,T,T)
    attn = s.matmul(v) # (B,H,T,T) (B,H,T,Dh) -> (B,H,T,Dh)
    attn = attn.transpose(1, 2).reshape(B, T, -1) # (B,H,T,Dh) -> (B,T,D)
    out = self.attn_o(attn) # (B,T,D) -> (B,T,D)
    return out

In [ ]:
B, T, D, H = 1, 2, 4, 2
assert D % H == 0, f'hidden D {D=} must be divisible by number of heads {H=}'
D_h = D // H
x = Tensor.arange(B*T*D).reshape(B,T,D).cast(dtypes.float).realize()

In [ ]:
with Stats("MHA Parameters:", header=True):
    model = MHA(D, H)
    for p in nn.state.get_parameters(model): p.realize()

with Stats("MHA Forward Pass:"):
    out = model(x).realize()

                                        Time                       Compute (FLOPs)            Memory                     Compute Bandwidth (FLOPS)  Memory Bandwidth           Arithmetic Intensity     
MHA Parameters:                                  142.08 ms,         14940 OPs,        2180 bytes,        105152.24 OPs/sec,       15343.50 bytes/sec,            6.85 OPs/byte
MHA Forward Pass:                                 28.41 ms,           356 OPs,         912 bytes,         12532.09 OPs/sec,       32104.67 bytes/sec,            0.39 OPs/byte


Why do the MHA parameters take up 268 bytes? Where does this number come from?
* The 4 model weights `self.attn_q`, `self.attn_k`, `self.attn_v`, `self.attn_o` are all matrices of shape $(D, D)$ with precision `float32`, i.e. $p=4$ bytes. This takes up $4 D^2 p = 256$ bytes.
* The 3 parameters `self.num_heads`, `self.head_dim`, `self.max_context` are all are integers with precision `int32`, i.e. `p=4` bytes. This takes up $3p = 12$ bytes.
* (256 bytes from model weights) + (12 bytes from the integer params) = 268 bytes, the exact number we got!

In [ ]:
param_bytes = sum(x.numel() * x.dtype.itemsize for x in nn.state.get_parameters(model))
assert param_bytes == 256

Why does the MHA forward pass take 572 OPs?

To better understand it, we can add more fine grained stats to the model. We must carefully add realizes after each call to stat for the memory to actually be allocated and ops actually computed.

In [ ]:
# multi-headed attention
# https://arxiv.org/abs/1706.03762
class MHA_Verbose:
  def __init__(self, dim:int, num_heads:int, max_context:int=0):

    self.num_heads = num_heads # H
    self.head_dim = dim // num_heads # Dh
    self.max_context = max_context # T

    self.attn_q = nn.Linear(dim, dim, bias=False) # (D,D)
    self.attn_k = nn.Linear(dim, dim, bias=False) # (D,D)
    self.attn_v = nn.Linear(dim, dim, bias=False) # (D,D)
    self.attn_o = nn.Linear(dim, dim, bias=False) # (D,D)

  def __call__(self, x:Tensor, start_pos:int=0, is_causal=False, kv_cache=False, rope=False) -> Tensor:
    # projections
    with Stats("q:"): q = self.attn_q(x).realize() # (B,T,D) -> (B,T,D)
    with Stats("k:"): k = self.attn_k(x).realize() # (B,T,D) -> (B,T,D)
    with Stats("v:"): v = self.attn_v(x).realize() # (B,T,D) -> (B,T,D)

    # reshape
    B, T, _ = x.shape
    q = q.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)  # (B,T,D) -> (B,H,T,Dh)
    k = k.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)  # (B,T,D) -> (B,H,T,Dh)
    v = v.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)  # (B,T,D) -> (B,H,T,Dh)

    # positional embeddings
    if rope:
      with Stats("rope:"): q, k = apply_rope(q, start_pos).realize(), apply_rope(k, start_pos).realize() # (B,H,T,Dh) -> (B,H,T,Dh)

    # kv cache
    if kv_cache and T > 1:
      with Stats("kv cache:"):
        if not hasattr(self, "kv_cache"):
          self.kv_cache = Tensor.zeros(2, self.max_context, B, self.num_heads, self.head_dim, dtype=k.dtype, device=k.device).contiguous().realize()
        self.kv_cache[:, start_pos:start_pos+T, :, :, :].assign(Tensor.stack(k, v)).realize()
        k = self.kv_cache[0, 0:start_pos+T, :, :, :].realize()
        v = self.kv_cache[1, 0:start_pos+T, :, :, :].realize()

    # compute attention
    with Stats("qk:"): qk = (q.matmul(k.transpose(-1, -2)) / math.sqrt(q.shape[-1])).realize() # (B,H,T,Dh) (B,H,Dh,T) -> (B,H,T,T)
    if is_causal and T > 1:
      mask = Tensor.full((1, 1, T, start_pos+T), float("-inf"), dtype=x.dtype, device=x.device).triu(start_pos+1) # (B,H,T,T)
      qk = (qk + mask).realize() # (B,H,T,T)
    with Stats("softmax:"): s = qk.softmax(-1).realize() # (B,H,T,T) -> (B,H,T,T)
    with Stats("attn:"): attn = s.matmul(v).realize() # (B,H,T,T) (B,H,T,Dh) -> (B,H,T,Dh)
    attn = attn.transpose(1, 2).reshape(B, T, -1).realize() # (B,H,T,Dh) -> (B,T,D)
    with Stats("out:"): out = self.attn_o(attn).realize() # (B,T,D) -> (B,T,D)
    return out

In [ ]:
print(f"{'':50}{'Time':<18}  {'Compute (FLOPs)':<18}  {'Memory':<18}  {'Compute Bandwidth (FLOPS)':<25}  {'Memory Bandwidth':<25}  {'Arithmetic Intensity':<25}")
print("MHA Parameters:")
with Stats("total:"):
    model = MHA_Verbose(D, H)
    Tensor.realize(*nn.state.get_parameters(model))

print("\nMHA Forward Pass:")
with Stats("total:"):
    out = model(x).realize()

                                                  Time                Compute (FLOPs)     Memory              Compute Bandwidth (FLOPS)  Memory Bandwidth           Arithmetic Intensity     
MHA Parameters:
total:                                           151.05 ms,         14396 OPs,        2100 bytes,         95303.27 OPs/sec,       13902.26 bytes/sec,            6.86 OPs/byte

MHA Forward Pass:
q:                                              829.58 us,            56 OPs,         128 bytes,         67503.79 OPs/sec,      154294.39 bytes/sec,            0.44 OPs/byte
k:                                              725.54 us,            56 OPs,         128 bytes,         77183.68 OPs/sec,      176419.84 bytes/sec,            0.44 OPs/byte
v:                                              537.38 us,            56 OPs,         128 bytes,        104210.28 OPs/sec,      238194.93 bytes/sec,            0.44 OPs/byte
qk:                                             924.88 us,            32 OPs, 

* **Query projection**: Computing `q = self.attn_q(x)` where `x.shape = (B,T,D)` and `attn_q.weight.shape = (D,D)` produces `q.shape = (B,T,D)`. The matrix multiplication requires $BTD(2D-1) = 57$ OPs.

* **Key projection**: Computing `k = self.attn_k(x)` similarly requires $BTD(2D-1) = 56$ OPs.

* **Value projection**: Computing `v = self.attn_v(x)` similarly requires $BTD(2D-1) = 56$ OPs.

* **Attention scores**: Computing `qk = q.matmul(k.transpose(-1, -2)) / math.sqrt(D_h)` where `q.shape = (B,H,T,D_h)` and `k.transpose(-1, -2).shape = (B,H,D_h,T)` produces `qk.shape = (B,H,T,T)`. The matrix multiplication requires $BHT^2(2D_h-1)$ OPs and the element-wise division requires $BHT^2$ OPs, totaling $BHT^2(2D_h) = 32$ OPs.

* **Softmax**: Computing `s = qk.softmax(-1)` where `qk.shape = (B,H,T,T)` produces `s.shape = (B,H,T,T)`. Estimating the FLOPs of softmax is a bit tricky. For numerical stability, softmax is computed as `exp(qk - qk_max) / sum(exp(qk - qk_max))`. Since the softmax is computed over the last dimension, we view `qk` as a tensor that contains $BHT$ independent rows (from dims 0-3) of length $T$ (from last dim):
    * **Max**: Finding `qk_max = max(qk)` per row: $T$ operations × $BHT$ rows = $BHT \cdot T$ OPs.
    * **Subtraction**: Computing `qk - qk_max` per element: $T$ operations × $BHT$ rows = $BHT \cdot T$ OPs.
    * **Exponentiation**: Computing `exp(qk - qk_max)`: $6T$ operations × $BHT$ rows = $BHT \cdot 6T$ OPs (each `exp` requires 6 OPs in tinygrad on Metal).
    * **Sum**: Summing $T$ values per row: $(T-1)$ operations × $BHT$ rows = $BHT \cdot (T-1)$ OPs.
    * **Division**: Dividing $T$ elements per row: $T$ operations × $BHT$ rows = $BHT \cdot T$ OPs.
    * **Total**: $BHT(10T - 1) = $ $76$ OPs.

* **Attention output**: Computing `attn = s.matmul(v)` where `s.shape = (B,H,T,T)` and `v.shape = (B,H,T,D_h)` produces `attn.shape = (B,H,T,D_h)`. This matrix multiplication requires $BHTD_h(2T-1) = $ $24$ OPs.

* **Output projection**: Computing `out = self.attn_o(attn)` where `attn.shape = (B,T,D)` and `attn_o.weight.shape = (D,D)` produces `out.shape = (B,T,D)`. This matrix multiplication requires $BTD(2D-1) = $ $56$ OPs.

# Group Query Attention

In [ ]:
# group-query attention
# https://arxiv.org/abs/2305.13245
class GQA:
  def __init__(self, D:int, H:int, Hkv:int):
    self.num_heads = H # number of heads
    self.num_headskv = Hkv # number of kv heads
    self.head_dim = D // H # head dimension

    self.attn_q = nn.Linear(D, self.head_dim*H, bias=False) # (D,Dh*H)
    self.attn_k = nn.Linear(D, self.head_dim*Hkv, bias=False) # (D,Dh*Hkv)
    self.attn_v = nn.Linear(D, self.head_dim*Hkv, bias=False) # (D,Dh*Hkv)
    self.attn_o = nn.Linear(D, self.head_dim*H, bias=False) # (D,Dh*H)

  def __call__(self, x:Tensor, start_pos:int=0, is_causal=True) -> Tensor:
    # projections
    q = self.attn_q(x) # (B,T,D) -> (B,T,Dh*H)
    k = self.attn_k(x) # (B,T,D) -> (B,T,Dh*Hkv)
    v = self.attn_v(x) # (B,T,D) -> (B,T,Dh*Hkv)

    # reshape
    B, T, _ = x.shape
    q = q.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)  # (B,T,D) -> (B,H,T,Dh)
    k = k.reshape(B, T, self.num_headskv, self.head_dim).transpose(1, 2)  # (B,T,Dh*Hkv) -> (B,Hkv,T,Dh)
    v = v.reshape(B, T, self.num_headskv, self.head_dim).transpose(1, 2)  # (B,T,Dh*Hkv) -> (B,Hkv,T,Dh)

    # gqa reshape
    k = k.repeat_interleave(self.num_heads // k.shape[-3], dim=-3) # (B,Hkv,T,Dh) -> (B,H,T,Dh)
    v = v.repeat_interleave(self.num_heads // v.shape[-3], dim=-3) # (B,Hkv,T,Dh) -> (B,H,T,Dh)

    # positional embeddings
    q, k = apply_rope(q, start_pos), apply_rope(k, start_pos) # (B,Hkv,T,Dh) -> (B,Hkv,T,Dh)

    # compute attention
    qk = q.matmul(k.transpose(-1, -2)) / math.sqrt(q.shape[-1]) # (B,H,T,Dh) (B,H,T,Dh) -> (B,H,T,T)
    if is_causal:
      mask = Tensor.full((1, 1, T, start_pos+T), float("-inf"), dtype=x.dtype, device=x.device).triu(start_pos+1) if T > 1 else None # (B,H,T,T)
      qk = qk + mask # (B,H,T,T)
    s = qk.softmax(-1) # (B,H,T,T) -> (B,H,T,T)
    attn = s.matmul(v) # (B,H,T,T) (B,H,T,Dh) -> (B,H,T,Dh)
    attn = attn.transpose(1, 2).reshape(B, T, -1) # (B,H,T,Dh) -> (B,T,D)
    out = self.attn_o(attn) # (B,T,D) -> (B,T,D)
    return out

In [ ]:
B, T, D, H, Hkv = 2, 32, 256, 4, 2
assert D % H == 0, f'hidden D {D=} must be divisible by number of heads {H=}'
x = Tensor.arange(B*T*D).reshape(B,T,D).cast(dtypes.float).realize()

with Stats("GQA Weights:", header=True):
    model = GQA(D, H, Hkv)
    for p in nn.state.get_parameters(model): p.realize()

with Stats("GQA Forward Pass:"):
    out = model(x).realize()

                                        Time                       Compute (FLOPs)            Memory                     Compute Bandwidth (FLOPS)  Memory Bandwidth           Arithmetic Intensity     
GQA Weights:                                     271.12 ms,      43819012 OPs,            6.29 MB,           161.62 MOPs/sec,              23.21 MB/sec,            6.96 OPs/byte
GQA Forward Pass:                                258.42 ms,      27657024 OPs,            2.25 MB,           107.03 MOPs/sec,               8.71 MB/sec,           12.29 OPs/byte


In [ ]:
# multi-query attention
# https://arxiv.org/abs/1911.02150
class MQA(GQA):
  def __init__(self, D:int, H:int):
    # MQA is just like GQA but we set set n_kv_heads to 1
    super().__init__(D, H, 1)

In [ ]:
B, T, D, H = 2, 32, 256, 4
assert D % H == 0, f'hidden D {D=} must be divisible by number of heads {H=}'

x = Tensor.arange(B*T*D).reshape(B,T,D).cast(dtypes.float).contiguous()
model = MQA(D, H)
out = model(x)
out.realize()

<Tensor <LB METAL (2, 32, 256) float ShapeTracker(views=(View(shape=(2, 32, 256), strides=(8192, 256, 1), offset=0, mask=None, contiguous=True),))> on METAL with grad None>

In [ ]:
B*H*T * ((c + 4)*T - 2)

NameError: name 'c' is not defined

In [ ]:
76 / (B * H * T)

19.0

In [ ]:
21/T-4

6.5